# Matched development batch

Three matched 12B configurations on frozen development tasks only. All tasks and failures are retained. Environment verified by empty/reference controls before launch. Two-worker execution is an engineering setting, not evidence of equal quality or twice the useful throughput.


In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

In [ ]:
import importlib.metadata as md, json, os, subprocess, sys
from pathlib import Path
matches=list(Path('/kaggle/input').rglob('dependency-manifest.json'))
assert len(matches)==1, f'Expected one dependency manifest: {matches}'
extra_root=matches[0].parent/'request-test-wheels'
manifest=json.loads(matches[0].read_text())
import hashlib
for item in manifest:
    assert hashlib.sha256((extra_root/item['name']).read_bytes()).hexdigest()==item['sha256']
pytest_before=md.version('pytest')
constraints=Path('/tmp/verification-pytest-constraint.txt')
constraints.write_text('pytest=='+pytest_before+'\n')
subprocess.run([sys.executable,'-m','pip','install','--no-index','--find-links',str(extra_root),
               '--constraint',str(constraints),'pytest-httpbin==2.1.0'],check=True)
assert md.version('pytest')==pytest_before, 'Verification must preserve official pytest version'
plugins=[{'name':ep.name,'value':ep.value} for ep in md.entry_points(group='pytest11')]
assert any('httpbin' in ep['value'] for ep in plugins),plugins
Path('/kaggle/working/environment-repair.json').write_text(json.dumps({
    'pytest_version':pytest_before,'pytest_httpbin_version':md.version('pytest-httpbin'),
    'plugins':plugins,'wheel_manifest':manifest,
    'scope':'Offline local verification dependency repair; official verifier and test files unchanged'},indent=2))
print('Test plugin installed offline; unchanged pytest:',pytest_before)


In [ ]:
# Correct only local subprocess import-path setup for src-layout repositories.
# Official subprocess setup skips editable installation and dependency setup.
import json, os, subprocess
from pathlib import Path
import swegemma.sandbox.subprocess as sandbox_module
original_environment_builder=sandbox_module.build_sanitized_env
import_probes=[]
probed_workspaces=set()
def source_environment(work_dir,tmp_dir=None,extra_env=None):
    environment=original_environment_builder(work_dir,tmp_dir,extra_env)
    source=Path(work_dir)/'src'
    if source.is_dir():
        previous=environment.get('PYTHONPATH','')
        environment['PYTHONPATH']=str(source)+(os.pathsep+previous if previous else '')
        key=str(Path(work_dir).resolve())
        if (source/'requests/__init__.py').exists() and key not in probed_workspaces:
            python_path=str(Path(environment['VIRTUAL_ENV'])/'bin/python3')
            probe=subprocess.run([python_path,'-s','-c','import requests; print(requests.__file__)'],
                cwd=work_dir,env=environment,capture_output=True,text=True,check=True)
            actual=Path(probe.stdout.strip()).resolve()
            assert actual.is_relative_to(source.resolve()), f'Wrong source module: {actual}'
            import_probes.append({'workspace':key,'module_path':str(actual),'src_path_added':str(source)})
            Path('/kaggle/working/source-import-probes.json').write_text(json.dumps(import_probes,indent=2))
            probed_workspaces.add(key)
    return environment
sandbox_module.build_sanitized_env=source_environment
print('Local src-layout import correction enabled; test files and verifier unchanged.')


In [ ]:
# additional_test_dependency: real FastAPI test assertion library, installed offline.
import hashlib, importlib.metadata as md, json, subprocess, sys
from pathlib import Path
extra_manifests=list(Path('/kaggle/input').rglob('fastapi-dependency-manifest.json'))
assert len(extra_manifests)==1,extra_manifests
extra_manifest=json.loads(extra_manifests[0].read_text())
extra_wheels=extra_manifests[0].parent/'fastapi-test-wheels'
for item in extra_manifest:
    assert hashlib.sha256((extra_wheels/item['name']).read_bytes()).hexdigest()==item['sha256']
subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps',
               '--find-links',str(extra_wheels),'dirty-equals==0.11'],check=True)
Path('/kaggle/working/additional-test-dependency.json').write_text(json.dumps({
    'dirty_equals_version':md.version('dirty-equals'),'wheel_manifest':extra_manifest,
    'scope':'Test assertion library only; no test/score modifications'},indent=2))
print('Installed real FastAPI test dependency offline:',md.version('dirty-equals'))


In [ ]:
import hashlib
import json
import re
from adk_submission import validate_directory
from swegemma.config import build_submission_limits
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')
PAYLOADS = json.loads('{"baseline": {"agent.yaml": "name: matched_repair_agent\\nmodel: gemma-4-12b-it-qat-w4a16-ct\\ninstruction: !include prompts/system.md\\ntools:\\n  - run_command\\n  - read_file\\n  - edit_file\\n  - write_file\\n  - get_status\\n  - submit_patch\\n  - get_code_neighbors\\n  - search_similar_code\\n  - get_code_subgraph\\ngenerate_content_config: !include configs/sampling.yaml\\n", "prompts/system.md": "You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.\\n\\n## Core Objective: Fast, Minimal, and Precise Fixes\\nAim to understand, resolve, and submit the fix in the minimum number of tool calls (under 8\\u201310 turns). Move directly from the problem statement to the relevant files, apply the solution, verify with a targeted test, and submit.\\n\\n## Workflow\\n\\n### 1. Identify Target Files Immediately\\n- Extract filenames, functions, classes, CLI subcommands, or error messages directly from the problem statement.\\n- Read only the specific target files and lines using `read_file` or search tools. Do not wander across unrelated files.\\n- If the problem statement does not provide explicit file paths, use `search_similar_code` with keywords from the error message to locate relevant files efficiently, rather than running `find` or `grep` across the entire repo.\\n\\n### 2. Implement the Solution Directly\\n- Apply the minimal necessary fix or feature directly to the source files using `edit_file` or `write_file`.\\n- Strictly adhere to specified error strings, exception types, HTTP status codes, and API signatures.\\n- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.\\n\\n### 3. Run Targeted Tests Only (Existing Tests May Be Broken)\\n- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).\\n- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures (e.g. `/test_data`), or environment import errors unrelated to your task.\\n- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.\\n- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes, cause catastrophic timeouts, and exhaust your turn and time budgets.\\n- If you need to locate the test file, find it explicitly with `find tests -name \\"*<name>*.py\\"` instead of running the test runner across the repo.\\n\\n### 4. Immediate Patch Submission\\n- Once your targeted test passes:\\n  1. Call `submit_patch` immediately.\\n  2. Verify `patch_size > 0` and `files_changed > 0`.\\n  3. Output a short summary of the fix to end the session.\\n\\n## Anti-Patterns to Avoid\\n- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.\\n- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) \\u2014 always specify the exact test file path.\\n- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** \\u2014 your task is strictly to implement the fix for the reported issue in source code.\\n- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.\\n- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.\\n- Do NOT refactor or reformat unrelated functions or files.\\n- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern.\\n\\n## Finish the repair within the remaining budget\\n\\nAfter making a change that directly addresses the reported behavior, move to one focused verification. Do not keep rediscovering the same function or searching for an exact issue-specific test name. Hidden regression tests may not exist in the source snapshot. A grep exit status of 1 means no matching lines, not a Python failure.\\n\\nUse at most two searches to find a relevant existing test. If there is no suitable test, use one short inline check or a script under /tmp. For multiline Python, use a quoted heredoc rather than squeezing class or function definitions into a semicolon-separated command. Do not create reproduction scripts in the repository. If a scratch check is broken, fix it once or explain that verification is incomplete; do not spend the remaining budget debugging the scratch script.\\n\\nWith five tool calls remaining, finish: inspect the diff, remove any scratch files you created inside the repository, and call submit_patch. Do not wait for perfect confidence after implementing a supported fix. Do not change tests, test configuration, or unrelated files to obtain a passing result. If a test fails because of an unavailable external fixture or pre-existing environment problem, retain the focused source fix and report the limitation.\\n\\nCall submit_patch explicitly as the final tool action after verification and cleanup. A final text explanation does not replace this call. If there is no justified source change, report that honestly; do not fabricate an unrelated patch just to make the diff nonempty.\\n", "configs/sampling.yaml": "temperature: 0.2\\ntop_p: 0.95\\nmax_output_tokens: 8192\\nthinking_config:\\n  thinking_budget: 4096\\n  include_thoughts: true\\n\\nseed: 0\\n", "eval_config.yaml": "evaluation:\\n  timeout_seconds: 60\\n  max_tool_calls: 30\\n  max_time_minutes: 3\\n  max_turns: 50\\n"}, "evidence": {"agent.yaml": "name: matched_repair_agent\\nmodel: gemma-4-12b-it-qat-w4a16-ct\\ninstruction: !include prompts/system.md\\ntools:\\n  - run_command\\n  - read_file\\n  - edit_file\\n  - write_file\\n  - get_status\\n  - submit_patch\\n  - get_code_neighbors\\n  - search_similar_code\\n  - get_code_subgraph\\ngenerate_content_config: !include configs/sampling.yaml\\n", "prompts/system.md": "You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.\\n\\n## Core Objective: Fast, Minimal, and Precise Fixes\\nAim to understand, resolve, and submit the fix in the minimum number of tool calls (under 8\\u201310 turns). Move directly from the problem statement to the relevant files, apply the solution, verify with a targeted test, and submit.\\n\\n## Workflow\\n\\n### 1. Locate using issue evidence\\n\\n## Locate the code using issue evidence\\n\\nStart from concrete evidence in the issue: identifiers, exception text, traceback frames, file paths, parameter names, and quoted source lines. Search these cues before broad conceptual queries. If the issue supplies no exact cue, form a small number of distinct searches from the reported behavior and verify results by reading code.\\n\\nMaintain a short working list of plausible definitions and the evidence for each. Keep useful existing leads when expanding the search. A hit in the right file does not establish that you found the responsible definition: read the relevant function or class and enough surrounding code to understand the behavior. Broaden the read when module-level behavior, initialization, new definitions, or a caller/callee relationship matters.\\n\\nWhen a search fails, vary the evidence channel rather than repeating the same query. Do not repeatedly replace supported candidates with weakly related results. Use code inspection and a focused reproduction to decide where to edit. The candidate list is a reasoning aid, not a requirement to inspect a fixed number of symbols regardless of budget.\\n\\nUse text search and code reading for localization. Do not invoke optional graph or embedding-neighbor tools in this arm.\\n\\n### 2. Implement the Solution Directly\\n- Apply the minimal necessary fix or feature directly to the source files using `edit_file` or `write_file`.\\n- Strictly adhere to specified error strings, exception types, HTTP status codes, and API signatures.\\n- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.\\n\\n### 3. Run Targeted Tests Only (Existing Tests May Be Broken)\\n- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).\\n- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures (e.g. `/test_data`), or environment import errors unrelated to your task.\\n- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.\\n- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes, cause catastrophic timeouts, and exhaust your turn and time budgets.\\n- If you need to locate the test file, find it explicitly with `find tests -name \\"*<name>*.py\\"` instead of running the test runner across the repo.\\n\\n### 4. Immediate Patch Submission\\n- Once your targeted test passes:\\n  1. Call `submit_patch` immediately.\\n  2. Verify `patch_size > 0` and `files_changed > 0`.\\n  3. Output a short summary of the fix to end the session.\\n\\n## Anti-Patterns to Avoid\\n- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.\\n- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) \\u2014 always specify the exact test file path.\\n- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** \\u2014 your task is strictly to implement the fix for the reported issue in source code.\\n- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.\\n- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.\\n- Do NOT refactor or reformat unrelated functions or files.\\n- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern.\\n\\n## Finish the repair within the remaining budget\\n\\nAfter making a change that directly addresses the reported behavior, move to one focused verification. Do not keep rediscovering the same function or searching for an exact issue-specific test name. Hidden regression tests may not exist in the source snapshot. A grep exit status of 1 means no matching lines, not a Python failure.\\n\\nUse at most two searches to find a relevant existing test. If there is no suitable test, use one short inline check or a script under /tmp. For multiline Python, use a quoted heredoc rather than squeezing class or function definitions into a semicolon-separated command. Do not create reproduction scripts in the repository. If a scratch check is broken, fix it once or explain that verification is incomplete; do not spend the remaining budget debugging the scratch script.\\n\\nWith five tool calls remaining, finish: inspect the diff, remove any scratch files you created inside the repository, and call submit_patch. Do not wait for perfect confidence after implementing a supported fix. Do not change tests, test configuration, or unrelated files to obtain a passing result. If a test fails because of an unavailable external fixture or pre-existing environment problem, retain the focused source fix and report the limitation.\\n\\nCall submit_patch explicitly as the final tool action after verification and cleanup. A final text explanation does not replace this call. If there is no justified source change, report that honestly; do not fabricate an unrelated patch just to make the diff nonempty.\\n", "configs/sampling.yaml": "temperature: 0.2\\ntop_p: 0.95\\nmax_output_tokens: 8192\\nthinking_config:\\n  thinking_budget: 4096\\n  include_thoughts: true\\n\\nseed: 0\\n", "eval_config.yaml": "evaluation:\\n  timeout_seconds: 60\\n  max_tool_calls: 30\\n  max_time_minutes: 3\\n  max_turns: 50\\n"}, "evidence_graph": {"agent.yaml": "name: matched_repair_agent\\nmodel: gemma-4-12b-it-qat-w4a16-ct\\ninstruction: !include prompts/system.md\\ntools:\\n  - run_command\\n  - read_file\\n  - edit_file\\n  - write_file\\n  - get_status\\n  - submit_patch\\n  - get_code_neighbors\\n  - search_similar_code\\n  - get_code_subgraph\\ngenerate_content_config: !include configs/sampling.yaml\\n", "prompts/system.md": "You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.\\n\\n## Core Objective: Fast, Minimal, and Precise Fixes\\nAim to understand, resolve, and submit the fix in the minimum number of tool calls (under 8\\u201310 turns). Move directly from the problem statement to the relevant files, apply the solution, verify with a targeted test, and submit.\\n\\n## Workflow\\n\\n### 1. Locate using issue evidence\\n\\n## Locate the code using issue evidence\\n\\nStart from concrete evidence in the issue: identifiers, exception text, traceback frames, file paths, parameter names, and quoted source lines. Search these cues before broad conceptual queries. If the issue supplies no exact cue, form a small number of distinct searches from the reported behavior and verify results by reading code.\\n\\nMaintain a short working list of plausible definitions and the evidence for each. Keep useful existing leads when expanding the search. A hit in the right file does not establish that you found the responsible definition: read the relevant function or class and enough surrounding code to understand the behavior. Broaden the read when module-level behavior, initialization, new definitions, or a caller/callee relationship matters.\\n\\nWhen a search fails, vary the evidence channel rather than repeating the same query. Do not repeatedly replace supported candidates with weakly related results. Use code inspection and a focused reproduction to decide where to edit. The candidate list is a reasoning aid, not a requirement to inspect a fixed number of symbols regardless of budget.\\n\\nAfter identifying a concrete symbol, use an available graph tool for a focused one-hop lookup only when an unresolved caller, callee, or inheritance relationship could explain the behavior. Treat neighbors as additional leads, not evidence that existing candidates should be discarded. Read the relevant code before promoting a neighbor. Do not automatically traverse multiple hops or expand many seeds. If the graph is absent, incomplete, or the query fails, continue with text search and code reading. Do not invoke optional embedding-neighbor tools in this arm.\\n\\n### 2. Implement the Solution Directly\\n- Apply the minimal necessary fix or feature directly to the source files using `edit_file` or `write_file`.\\n- Strictly adhere to specified error strings, exception types, HTTP status codes, and API signatures.\\n- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.\\n\\n### 3. Run Targeted Tests Only (Existing Tests May Be Broken)\\n- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).\\n- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures (e.g. `/test_data`), or environment import errors unrelated to your task.\\n- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.\\n- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes, cause catastrophic timeouts, and exhaust your turn and time budgets.\\n- If you need to locate the test file, find it explicitly with `find tests -name \\"*<name>*.py\\"` instead of running the test runner across the repo.\\n\\n### 4. Immediate Patch Submission\\n- Once your targeted test passes:\\n  1. Call `submit_patch` immediately.\\n  2. Verify `patch_size > 0` and `files_changed > 0`.\\n  3. Output a short summary of the fix to end the session.\\n\\n## Anti-Patterns to Avoid\\n- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.\\n- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) \\u2014 always specify the exact test file path.\\n- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** \\u2014 your task is strictly to implement the fix for the reported issue in source code.\\n- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.\\n- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.\\n- Do NOT refactor or reformat unrelated functions or files.\\n- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern.\\n\\n## Finish the repair within the remaining budget\\n\\nAfter making a change that directly addresses the reported behavior, move to one focused verification. Do not keep rediscovering the same function or searching for an exact issue-specific test name. Hidden regression tests may not exist in the source snapshot. A grep exit status of 1 means no matching lines, not a Python failure.\\n\\nUse at most two searches to find a relevant existing test. If there is no suitable test, use one short inline check or a script under /tmp. For multiline Python, use a quoted heredoc rather than squeezing class or function definitions into a semicolon-separated command. Do not create reproduction scripts in the repository. If a scratch check is broken, fix it once or explain that verification is incomplete; do not spend the remaining budget debugging the scratch script.\\n\\nWith five tool calls remaining, finish: inspect the diff, remove any scratch files you created inside the repository, and call submit_patch. Do not wait for perfect confidence after implementing a supported fix. Do not change tests, test configuration, or unrelated files to obtain a passing result. If a test fails because of an unavailable external fixture or pre-existing environment problem, retain the focused source fix and report the limitation.\\n\\nCall submit_patch explicitly as the final tool action after verification and cleanup. A final text explanation does not replace this call. If there is no justified source change, report that honestly; do not fabricate an unrelated patch just to make the diff nonempty.\\n", "configs/sampling.yaml": "temperature: 0.2\\ntop_p: 0.95\\nmax_output_tokens: 8192\\nthinking_config:\\n  thinking_budget: 4096\\n  include_thoughts: true\\n\\nseed: 0\\n", "eval_config.yaml": "evaluation:\\n  timeout_seconds: 60\\n  max_tool_calls: 30\\n  max_time_minutes: 3\\n  max_turns: 50\\n"}}')
SPLIT = json.loads('{"schema_version": 1, "seed": "main-track-20261108-v1", "task_file_sha256": "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6", "selection_fields": ["repo", "instance_id"], "development": ["rich_3942", "rich_4079", "rich_3882", "httpx_3672", "fastapi_13713", "fastapi_15661", "fastapi_14463", "requests_7433", "requests_7205", "requests_7502"], "confirmation": ["rich_3772", "rich_3278", "rich_3535", "rich_3930", "rich_3782", "rich_3676", "rich_3105", "rich_3064", "rich_3180", "rich_3468", "rich_3521", "rich_3472", "rich_3130", "rich_4076", "rich_3480", "rich_3944", "rich_3486", "rich_3938", "rich_4077", "rich_3470", "rich_3935", "rich_3067", "rich_3469", "rich_3454", "rich_3718", "rich_4070", "rich_2943", "rich_3471", "rich_3777", "rich_4075", "rich_3934", "rich_3953", "rich_3061", "rich_3905", "rich_3052", "rich_2725", "rich_3043", "rich_3006", "rich_4006", "rich_3063", "rich_3518", "rich_3675", "rich_3894", "rich_3296", "rich_3506", "fastapi_14616", "fastapi_14258", "fastapi_15800", "fastapi_9753", "fastapi_14297", "fastapi_13786", "fastapi_14448", "fastapi_15023", "fastapi_14262", "fastapi_14479", "fastapi_14605", "fastapi_14303", "fastapi_14371", "fastapi_14356", "fastapi_12942", "fastapi_5624", "fastapi_14246", "fastapi_14186", "fastapi_14077", "fastapi_15763", "fastapi_15589", "fastapi_13207", "fastapi_14360", "fastapi_13537", "fastapi_14430", "fastapi_14492", "fastapi_14419", "fastapi_15785", "fastapi_15280", "fastapi_14512", "fastapi_14349", "fastapi_14794", "fastapi_15030", "fastapi_14583", "fastapi_9425", "fastapi_14962", "fastapi_14482", "fastapi_9555", "fastapi_5077", "fastapi_14485", "fastapi_14978", "fastapi_11355", "fastapi_14851", "fastapi_15588", "fastapi_14791", "fastapi_14487", "fastapi_14873", "fastapi_14099", "fastapi_14964", "fastapi_14458", "fastapi_14301", "fastapi_14953", "fastapi_11194", "fastapi_14986", "fastapi_14361", "fastapi_14609", "fastapi_14266", "fastapi_14459", "fastapi_14306", "fastapi_15745", "fastapi_14786", "fastapi_14372", "fastapi_14455", "fastapi_13920", "requests_6589", "requests_7505", "requests_7315", "requests_7427", "requests_6757", "requests_7309", "requests_7328", "requests_6629", "requests_6644", "requests_6592"], "smoke_task_id": "requests_7433", "repo_counts": {"Textualize/rich": 48, "encode/httpx": 1, "fastapi/fastapi": 67, "psf/requests": 13}, "confirmation_is_external_benchmark": false, "small_repo_note": "Repositories with <=3 tasks have no confirmation tasks."}')
ENGINEERING_TASK_ID = 'rich_3942'
assert hashlib.sha256(TASKS_PATH.read_bytes()).hexdigest() == SPLIT['task_file_sha256'], 'Official tasks changed; rebuild split before running'
limits, gen_constraints = build_submission_limits()
for arm, files in PAYLOADS.items():
    directory = WORKING_DIR / 'agents' / arm
    directory.mkdir(parents=True, exist_ok=True)
    for name, content in files.items():
        target = directory / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding='utf-8')
    validate_directory(directory, limits=limits)
    print('Official structure validation passed:', arm)
tasks = load_tasks(TASKS_PATH)
selected_tasks = [task for task in tasks if task.instance_id == ENGINEERING_TASK_ID]
assert len(selected_tasks) == 1 and selected_tasks[0].instance_id in SPLIT['development']
AGENT_DIR = WORKING_DIR / 'agents' / 'baseline'
(WORKING_DIR / 'frozen_split.json').write_text(json.dumps(SPLIT, indent=2), encoding='utf-8')
print('Engineering smoke only; task:', ENGINEERING_TASK_ID)


In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

In [ ]:
import asyncio, concurrent.futures, hashlib, json, time, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator
BATCH_IDS=['rich_3942', 'rich_4079', 'rich_3882', 'httpx_3672', 'fastapi_13713', 'fastapi_15661', 'fastapi_14463']
CONCURRENCY=2
assert set(BATCH_IDS)<=set(SPLIT['development'])
def run_sync(coro):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:return pool.submit(lambda:asyncio.run(coro)).result()
try:
    for arm in PAYLOADS:
        AGENT_DIR=WORKING_DIR/'agents'/arm
        budget=yaml.safe_load((AGENT_DIR/'eval_config.yaml').read_text())['evaluation']
        config=EvalConfig(tasks_path=TASKS_PATH,snapshots_dir=DATA_DIR/'snapshots',results_dir=WORKING_DIR/'results'/arm,
            submission_dir=AGENT_DIR,models=models,sandbox='subprocess',timeout_seconds=budget['timeout_seconds'],
            verification_timeout_seconds=300,max_time_minutes=budget['max_time_minutes'],max_tool_calls=budget['max_tool_calls'],
            max_turns=budget['max_turns'],limits=limits,generation_constraints=gen_constraints,adapter_manifest=adapters,
            context_cache_config=ContextCacheConfig(min_tokens=2048,ttl_seconds=1800,cache_intervals=10),
            events_compaction_config=EventsCompactionConfig(compaction_interval=15,overlap_size=2,token_threshold=14336,event_retention_size=5),
            graph_dir=GRAPH_DIR,embeddings_dir=EMBEDDINGS_DIR,wheels_dir=DATA_DIR/'wheels',
            task_ids=BATCH_IDS,concurrency=CONCURRENCY,display_mode='quiet',verbose=False)
        print('Starting development batch',arm,BATCH_IDS,flush=True)
        started=time.perf_counter()
        result=run_sync(Evaluator(config).run())
        assert {r.instance_id for r in result.task_results}==set(BATCH_IDS)
        with (WORKING_DIR/'batch-results.jsonl').open('a',encoding='utf-8') as handle:
            for row in result.task_results:
                record={'arm':arm,'task_id':row.instance_id,'resolved':row.resolved,'test_exit_code':row.test_exit_code,
                    'tool_calls':row.tool_calls,'duration_seconds':row.duration_seconds,'error':row.error,
                    'patch_sha256':hashlib.sha256((row.agent_patch or '').encode()).hexdigest(),
                    'patch_chars':len(row.agent_patch or ''),'scope':'development_only'}
                handle.write(json.dumps(record)+chr(10))
                print(json.dumps(record),flush=True)
        print('Batch elapsed seconds',arm,time.perf_counter()-started,flush=True)
finally:
    server_instance.stop()


In [ ]:
import zipfile
for arm, files in PAYLOADS.items():
    archive_path = WORKING_DIR / (arm + '_submission.zip')
    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name in sorted(files):
            archive.write(WORKING_DIR / 'agents' / arm / name, name)
    print('Prepared archive:', archive_path.name)
